# Select the Model With Best Hyper Pearameters Tuining

## Import Libraries

In [ ]:
# Data Manipulation
import pandas as pd
import numpy as np
import os


# Data Visualization
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import plotly.express as px


# Data Preprocessing
from sklearn.preprocessing import StandardScaler,MinMaxScaler,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split


# Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,mean_squared_error,r2_score


# GridSearchCV for  cross Validation
from sklearn.model_selection import GridSearchCV


# Ignore warnings
import warnings
warnings.filterwarnings("ignore")




In [ ]:
# Load the Data Set
df = pd.read_csv('/content/pakwheels_cars_cleaned.csv')

# New Section

In [ ]:
df.head()

,body_type,city,color,engine_cc,fuel,make,mileage_km,model,price,transmission,year
0,Sedan,Karachi,Black,2100.0,Petrol,Mercedes Benz,98000,E Class,9850000,Automatic,2011
1,Crossover,Yazman mandi,White,1000.0,Petrol,Toyota,28000,Raize,6190000,Automatic,2020
2,Hatchback,Lahore,Black,1500.0,Hybrid,Toyota,98213,Aqua,3290000,Automatic,2013
3,Sedan,Lahore,Grey,2000.0,Petrol,Hyundai,75987,Elantra,7500000,Automatic,2023
4,Hatchback,Karachi,Burgundy,1500.0,Hybrid,Toyota,89000,Aqua,5100000,Automatic,2020


In [ ]:
df.columns

Index(['body_type', 'city', 'color', 'engine_cc', 'fuel', 'make', 'mileage_km',
       'model', 'price', 'transmission', 'year'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1248 entries, 0 to 1247
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   body_type     1248 non-null   object 
 1   city          1248 non-null   object 
 2   color         1248 non-null   object 
 3   engine_cc     1248 non-null   float64
 4   fuel          1248 non-null   object 
 5   make          1248 non-null   object 
 6   mileage_km    1248 non-null   int64  
 7   model         1248 non-null   object 
 8   price         1248 non-null   int64  
 9   transmission  1248 non-null   object 
 10  year          1248 non-null   int64  
dtypes: float64(1), int64(3), object(7)
memory usage: 107.4+ KB


In [ ]:
df.describe(include= "all")

,body_type,city,color,engine_cc,fuel,make,mileage_km,model,price,transmission,year
count,1248,1248,1248,1248.000000,1248,1248,1248.000000,1248,1.248000e+03,1248,1248.000000
unique,18,82,61,NaN,6,33,NaN,146,NaN,2,NaN
top,Sedan,Lahore,White,NaN,Petrol,Toyota,NaN,Corolla,NaN,Automatic,NaN
freq,424,313,508,NaN,1092,377,NaN,147,NaN,771,NaN
mean,NaN,NaN,NaN,1388.101763,NaN,NaN,104554.403045,NaN,4.542186e+06,NaN,2014.411058
std,NaN,NaN,NaN,669.866106,NaN,NaN,82177.428857,NaN,6.432850e+06,NaN,8.107741
min,NaN,NaN,NaN,658.000000,NaN,NaN,1.000000,NaN,2.000000e+05,NaN,1965.000000
25%,NaN,NaN,NaN,1000.000000,NaN,NaN,50000.000000,NaN,1.900000e+06,NaN,2010.000000
50%,NaN,NaN,NaN,1300.000000,NaN,NaN,90000.000000,NaN,3.300000e+06,NaN,2016.000000
75%,NaN,NaN,NaN,1600.000000,NaN,NaN,138000.000000,NaN,4.725000e+06,NaN,2021.000000


In [ ]:
# One Hot encoding of categories having less than 20 categories
df = pd.get_dummies(df, columns=['body_type',"fuel","transmission",], drop_first=True)

In [ ]:
# Label encoding of categorical columns

categorical_cols = ['city','color', 'make', 'model']

le = LabelEncoder()
for cols in categorical_cols:
    df[cols] = le.fit_transform(df[cols])


In [ ]:
# Select Features and Variables
X =  df.drop('price', axis=1)
y= df['price']

In [ ]:
# Split the data set into train and test
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [ ]:
# Create a dictionary of model to evaluate models
%time

models = {
    "LinearRegression" : LinearRegression(),
    "SVR" : SVR(),
    "DecisionTreeRegressor" : DecisionTreeRegressor(),
    "KNeighborsRegressor" : KNeighborsRegressor(),
    "RandomForestRegressor" : RandomForestRegressor(),
    "GradientBoostingRegressor" : GradientBoostingRegressor(),
    "XGBRFRegressor" : XGBRegressor()
}


# train and predict each model with evaluation metrics as well making a for loop to iterate over the models


model_scores = {}
for name, model in models.items():
    # fit each model from models on training data
    model.fit(X_train, y_train)

    # make prediction from each model
    y_pred = model.predict(X_test)
    MSE = mean_squared_error(y_test, y_pred)
    R2_score = r2_score(y_test, y_pred)
    MAE = mean_absolute_error(y_test, y_pred)
    model_scores.setdefault(name, []).extend([MAE,MSE, R2_score])


# for name, metric in model_scores.items():
#     print(f"{name} Mean Absolute Error is :{metric[0]}")
#     print(f"{name} Mean Squared Error is :{metric[1]}")
#     print(f"{name} r2_score is :{metric[2]}")
#     print(f"{name} Mean Absolute Percentage Error is :{metric[3]}")
#     print("---------------------------------------------------------")




best_model_name = None
best_r2 = float("-inf")
best_mae = float("inf")

for name, metric in model_scores.items():
    mae = metric[0]
    r2 = metric[2]

    if r2 > best_r2 or (r2 == best_r2 and mae < best_mae):
        best_r2 = r2
        best_mae = mae
        best_model_name = name

print("====================================")
print(f"Best Model Selected: {best_model_name}")
print(f"Best R2 Score      : {best_r2}")
print(f"Best MAE           : {best_mae}")
print("====================================")

# Get the best trained model object
best_model = models[best_model_name]





CPU times: user 2 µs, sys: 0 ns, total: 2 µs
Wall time: 3.34 µs
Best Model Selected: GradientBoostingRegressor
Best R2 Score      : 0.5862670367681417
Best MAE           : 1167309.7466872667


In [1]:
%time

models = {
          'LinearRegression' : (LinearRegression(), {}),
          'SVR' : (SVR(), {'kernel': ['rbf', 'poly', 'sigmoid'], 'C': [0.1, 1, 10], 'gamma': [1, 0.1], 'epsilon': [0.1, 0.01]}),
          'DecisionTreeRegressor' : (DecisionTreeRegressor(), {'max_depth': [None, 5, 10], 'splitter': ['best', 'random']}),
          'RandomForestRegressor' : (RandomForestRegressor(), {'n_estimators': [10, 100], 'max_depth': [None, 5, 10]}),
          'KNeighborsRegressor' : (KNeighborsRegressor(), {'n_neighbors': np.arange(3, 100, 2), 'weights': ['uniform', 'distance']}),
          'GradientBoostingRegressor' : (GradientBoostingRegressor(), {'loss': ['ls', 'lad', 'huber', 'quantile'], 'n_estimators': [10, 100]}),
          'XGBRegressor' : (XGBRegressor(), {'n_estimators': [10, 100], 'learning_rate': [0.1, 0.01]}),
          }

# train and predict each model with evaluation metrics as well making a for loop to iterate over the models
model_scores = {}
for name, (model, params) in models.items():
    # create a pipline
    pipeline = GridSearchCV(model, params, cv=5)

    # fit the pipeline
    pipeline.fit(X_train, y_train)

    # make prediction from each model
    y_pred = pipeline.predict(X_test)
   # train and predict each model with evaluation metrics as well making a for loop to iterate over the models

    # make prediction from each model
    y_pred = pipeline.predict(X_test)
    MSE = mean_squared_error(y_test, y_pred)
    R2_score = r2_score(y_test, y_pred)
    MAE = mean_absolute_error(y_test, y_pred)
    model_scores.setdefault(name, []).extend([MAE,MSE, R2_score])


# for name, metric in model_scores.items():
#     print(f"{name} Mean Absolute Error is :{metric[0]}")
#     print(f"{name} Mean Squared Error is :{metric[1]}")
#     print(f"{name} r2_score is :{metric[2]}")
#     print(f"{name} Mean Absolute Percentage Error is :{metric[3]}")
#     print("---------------------------------------------------------")




best_model_name = None
best_r2 = float("-inf")
best_mae = float("inf")

for name, metric in model_scores.items():
    mae = metric[0]
    r2 = metric[2]

    if r2 > best_r2 or (r2 == best_r2 and mae < best_mae):
        best_r2 = r2
        best_mae = mae
        best_model_name = name

print("====================================")
print(f"Best Model Selected: {best_model_name}")
print(f"Best R2 Score      : {best_r2}")
print(f"Best MAE           : {best_mae}")
print("====================================")

# Get the best trained model object
best_model = models[best_model_name]






CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 8.11 µs


NameError: name 'LinearRegression' is not defined